# Chapter 3: Classical Statistics for Population Health## 3.1 Setting Up Your WorkbenchIn this section, we'll confirm that your Python environment is working by loading a realpublic health dataset and creating a simple visualization. By the end of this notebook,you'll have downloaded county-level health data from the County Health Rankings & Roadmapsproject and produced a scatter plot showing how diabetes prevalence relates to householdincome across counties in California.If this notebook runs without errors, your workbench is ready for the rest of the chapter.

### Installing and Importing LibrariesWe'll use three libraries throughout this book:- **pandas** — for loading and manipulating tabular data- **matplotlib** — for creating charts and visualizations- **scikit-learn** — for building statistical models (we won't use this yet, but let's  confirm it's installed)Run the cell below to install them. If they're already installed, pip will simply reportthat the requirement is satisfied.

In [ ]:
# Install the libraries we'll use throughout the book.# You only need to run this cell once per environment.%pip install pandas matplotlib scikit-learn numpy

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport sklearnprint(f"pandas     : {pd.__version__}")print(f"numpy      : {np.__version__}")print(f"matplotlib : {plt.matplotlib.__version__}")print(f"scikit-learn: {sklearn.__version__}")print("\nAll libraries loaded successfully!")

### About the DataThe [County Health Rankings & Roadmaps](https://www.countyhealthrankings.org/) project,a collaboration between the Robert Wood Johnson Foundation and the University of WisconsinPopulation Health Institute, publishes annual county-level health data for every county inthe United States. The dataset includes dozens of measures covering health outcomes, healthbehaviors, clinical care, social and economic factors, and the physical environment.We'll use several measures from this dataset throughout the chapter. The two we start withare:- **Diabetes prevalence** — the percentage of adults aged 20 and older with diagnosed diabetes- **Median household income** — the income level at which half of households earn more  and half earn lessThe data is freely available and requires no login or account to download.

### Downloading the DataThe County Health Rankings publishes its national analytic dataset as a CSV file each year.Before we can load it, you'll need to download it manually:1. Visit the County Health Rankings data documentation page at   [countyhealthrankings.org/health-data/methodology-and-sources/data-documentation](https://www.countyhealthrankings.org/health-data/methodology-and-sources/data-documentation)2. Find the **National Data** section and download the **analytic data** CSV file   (the filename will look something like `analytic_data2024.csv`)3. Place the downloaded file in a folder called `working` next to this notebookYour file structure should look like this:```chapter_3.ipynbworking/    analytic_data2024.csv```The CSV file has a two-row header: the first row contains variable codes (like `v060_rawvalue`),and the second row contains human-readable descriptions. We'll keep the variable codes asour column names and skip the description row.

In [ ]:
from pathlib import Pathimport glob# Look for the analytic data CSV in the working/ folder.# This pattern matches any file starting with "analytic_data" so it will# work regardless of which year's file you downloaded.csv_files = sorted(glob.glob("working/analytic_data*.csv"))if not csv_files:    raise FileNotFoundError(        "No analytic_data*.csv file found in the working/ folder.\n"        "Please download the file from countyhealthrankings.org and place it "        "in a folder called 'working' next to this notebook."    )csv_path = csv_files[-1]  # Use the most recent file if multiple are presentprint(f"Loading: {csv_path}")# Read the CSV. Row 0 has variable codes (our column names); row 1 has# descriptions that we skip.  encoding="latin-1" handles special characters# that occasionally appear in county names.chr_data = pd.read_csv(    csv_path,    header=0,    skiprows=[1],    encoding="latin-1",)print(f"Dataset shape: {chr_data.shape[0]:,} rows × {chr_data.shape[1]} columns")

### Exploring the DataBefore we do anything else, let's look at what we downloaded. This is a habit worthbuilding — always inspect your data before analyzing it. Even a quick glance canreveal missing values, unexpected formats, or columns with names you didn't expect.

In [ ]:
# Show the first few rows to get a feel for the data.chr_data.head()

In [ ]:
# List all column names. This is especially useful with large datasets# where .head() truncates the display.print("Number of columns:", len(chr_data.columns))print()for col in chr_data.columns:    print(col)

The County Health Rankings uses a coding scheme for its variable names. Each healthmeasure has a number, and the raw value for that measure is stored in a column named`v{number}_rawvalue`. The columns we need are:| Column | Description ||--------|-------------|| `statecode` | Two-digit state FIPS code || `state` | State abbreviation || `county` | County name || `v060_rawvalue` | Diabetes prevalence (proportion of adults 20+ with diagnosed diabetes) || `v063_rawvalue` | Median household income (dollars) |Let's confirm these columns exist and see what the values look like.

In [ ]:
# The columns we'll work with.DIABETES_COL = "v060_rawvalue"INCOME_COL = "v063_rawvalue"# Verify they exist in the dataset.for col in [DIABETES_COL, INCOME_COL, "state", "county"]:    if col in chr_data.columns:        print(f"  ✓  '{col}' found")    else:        print(f"  ✗  '{col}' NOT found — check the column list above")

In [ ]:
# Quick summary of our two key columns.chr_data[[DIABETES_COL, INCOME_COL]].describe()

### Filtering to CaliforniaThe full dataset contains every county in the nation. Throughout this chapter we'll focuson California — it has 58 counties spanning a wide range of both income levels and diabetesrates, from wealthy coastal counties to lower-income agricultural communities in theCentral Valley.Feel free to change the state abbreviation below to explore your own state.

In [ ]:
# Choose a state. Change this abbreviation to explore a different state.STATE = "CA"# Filter to the chosen state and drop rows where either measure is missing.# The first row for each state is a state-level summary (countycode == 0),# which we exclude so we only have individual counties.state_data = (    chr_data[        (chr_data["state"] == STATE)        & (chr_data["countycode"] != 0)    ]    .dropna(subset=[DIABETES_COL, INCOME_COL])    .copy())print(f"Counties in {STATE} with complete data: {len(state_data)}")

### Your First Visualization: Diabetes vs. IncomeNow for the payoff. We'll create a scatter plot with median household income on thex-axis and diabetes prevalence on the y-axis. Each dot represents one California county.Before you run the cell, take a moment to form a hypothesis: do you expect countieswith higher incomes to have higher or lower diabetes rates?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))ax.scatter(    state_data[INCOME_COL],    state_data[DIABETES_COL] * 100,  # Convert proportion to percentage    alpha=0.6,    edgecolors="steelblue",    facecolors="lightblue",    linewidths=0.8,)ax.set_xlabel("Median Household Income ($)", fontsize=12)ax.set_ylabel("Diabetes Prevalence (%)", fontsize=12)ax.set_title(    f"Diabetes Prevalence vs. Median Household Income\n"    f"California Counties",    fontsize=14,)# Format the x-axis labels as dollar amounts.ax.xaxis.set_major_formatter(    plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()

### What Do You See?If your environment is set up correctly, you should see a scatter plot showing adownward trend: counties with higher household incomes tend to have lower diabetesrates. This negative relationship is a pattern that public health researchers havedocumented extensively — income is one of the strongest social determinants of health.But notice that the dots don't fall on a perfect line. Some lower-income counties haverelatively low diabetes rates, and some higher-income counties have rates higher thanyou might expect. In the sections that follow, we'll learn how to quantify thisrelationship using regression, measure how well our model fits, and understand whyno single factor tells the whole story.### Workbench CheckIf you made it this far without errors, your environment is ready. Here's what we confirmed:1. **pandas** can load data from a local CSV file2. **matplotlib** can render charts in your notebook3. **scikit-learn** is installed (we'll use it starting in Section 3.3)4. You can load and filter the County Health Rankings datasetIn the next section, we'll dig deeper into this same dataset to understand howstatisticians describe and summarize population health data.

---## 3.2 Describing Population Health DataBefore fitting any model, we need to understand the shape and character of our data.In this section we'll compute summary statistics and plot distributions for severalhealth indicators across California's counties. We'll also build a correlation matrixto see which indicators tend to move together — a first step toward understanding whichfactors might be useful predictors.The additional columns we'll work with are:| Column | Description ||--------|-------------|| `v009_rawvalue` | Adult smoking rate (proportion of adults who currently smoke) || `v011_rawvalue` | Adult obesity rate (proportion of adults with BMI ≥ 30) || `v005_rawvalue` | Preventable hospital stays (rate per 100,000 Medicare enrollees) |

In [ ]:
# Define the health indicators we'll study in this section.INDICATORS = {    "v060_rawvalue": "Diabetes Prevalence",    "v009_rawvalue": "Adult Smoking Rate",    "v011_rawvalue": "Adult Obesity Rate",    "v005_rawvalue": "Preventable Hospital Stays",    "v063_rawvalue": "Median Household Income",}# Verify all columns are present.for col, label in INDICATORS.items():    if col in chr_data.columns:        print(f"  ✓  '{col}'  ({label})")    else:        print(f"  ✗  '{col}'  ({label}) — NOT found")

### Summary StatisticsWe'll start with the simplest question: what does a "typical" California county looklike for each of these measures? The pandas `describe()` method gives us the mean,standard deviation, min, max, and quartiles in one shot.

In [ ]:
# Build a clean dataframe of California county indicators.ca_indicators = (    state_data[list(INDICATORS.keys())]    .rename(columns=INDICATORS)    .copy())# Scale proportion columns to percentages for readability.# Preventable hospital stays is already a rate per 100,000, so leave it as-is.# Median household income is in dollars, so leave it as-is.pct_cols = ["Diabetes Prevalence", "Adult Smoking Rate", "Adult Obesity Rate"]ca_indicators[pct_cols] = ca_indicators[pct_cols] * 100ca_indicators.describe().round(2)

Look at the difference between the mean and the median (the 50th percentile) for eachvariable. When those two values are close together, the distribution is roughly symmetric.When they're far apart, the distribution is skewed — pulled in one direction by extremevalues. We can see this much more clearly with histograms.

### Distributions: HistogramsA histogram bins the data into ranges and counts how many counties fall into each bin.The shape of the histogram tells us about the distribution. A bell-shaped histogramindicates a roughly normal (Gaussian) distribution. A histogram with a long tail onone side indicates a skewed distribution.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))fig.suptitle("Distribution of Health Indicators — California Counties", fontsize=14)plot_cols = ["Diabetes Prevalence", "Adult Smoking Rate",             "Adult Obesity Rate", "Preventable Hospital Stays"]colors = ["steelblue", "seagreen", "coral", "mediumpurple"]for ax, col, color in zip(axes.flat, plot_cols, colors):    data = ca_indicators[col].dropna()    ax.hist(data, bins=15, color=color, edgecolor="white", alpha=0.8)    ax.axvline(data.mean(), color="black", linestyle="--", linewidth=1.2,               label=f"Mean: {data.mean():.1f}")    ax.axvline(data.median(), color="red", linestyle="-", linewidth=1.2,               label=f"Median: {data.median():.1f}")    ax.set_xlabel(col)    ax.set_ylabel("Number of Counties")    ax.legend(fontsize=9)plt.tight_layout()plt.show()

Notice which distributions look roughly symmetric (bell-shaped) and which are skewed.The dashed black line (mean) and solid red line (median) help you see the skew: whenthe mean is pulled to the right of the median, the distribution has a right skew,meaning a few counties with very high values are pulling the average up.Understanding the shape of your data matters because many statistical techniques —including ordinary linear regression — make assumptions about how the data is distributed.When those assumptions are violated, the results can be misleading.

### Correlation: Which Indicators Move Together?Correlation measures the strength and direction of a linear relationship between twovariables. A correlation of +1 means they move perfectly together, -1 means they movein exactly opposite directions, and 0 means no linear relationship.Let's compute the correlation matrix for all of our California county indicators.

In [ ]:
# Compute the correlation matrix.corr_matrix = ca_indicators.corr().round(3)# Display it as a heatmap.fig, ax = plt.subplots(figsize=(8, 6))im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")# Label the axes.labels = corr_matrix.columnsax.set_xticks(range(len(labels)))ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=10)ax.set_yticks(range(len(labels)))ax.set_yticklabels(labels, fontsize=10)# Annotate each cell with the correlation value.for i in range(len(labels)):    for j in range(len(labels)):        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}",                ha="center", va="center", fontsize=10,                color="white" if abs(corr_matrix.iloc[i, j]) > 0.6 else "black")plt.colorbar(im, ax=ax, label="Correlation", shrink=0.8)ax.set_title("Correlation Between Health Indicators — California Counties", fontsize=13)plt.tight_layout()plt.show()

Take a moment to study the heatmap. You should see strong positive correlations betweendiabetes, smoking, and obesity — counties where one is high tend to have the others highas well. Median household income should show a negative correlation with these healthmeasures, consistent with the scatter plot we created in section 3.1.These correlations give us our first hint about which factors might be useful predictorsin a regression model. In the next section, we'll formalize that idea.

---## 3.3 Linear Regression: Drawing the LineLet's return to the scatter plot of diabetes prevalence vs. median household income forCalifornia counties. There's clearly a downward trend — but can we quantify it? Can wedraw the "best" line through the data and use it to make predictions?Linear regression gives us a way to do exactly that. The model takes the form:$$y = mx + b$$where *y* is the outcome we want to predict (diabetes prevalence), *x* is the predictor(household income), *m* is the slope (how much *y* changes when *x* increases by one unit),and *b* is the intercept (the predicted *y* when *x* is zero).But what makes a line "best"? We need a way to measure how wrong a line is. The standardapproach is **mean squared error (MSE)**: for each county, compute the gap between theactual value and the predicted value (the **residual**), square it, and take the averageacross all counties. The best line is the one that minimizes this number.

In [ ]:
from sklearn.linear_model import LinearRegressionfrom sklearn.metrics import mean_squared_error, r2_score# Prepare the data. scikit-learn expects X as a 2D array.X = state_data[[INCOME_COL]].valuesy = state_data[DIABETES_COL].values * 100  # percentage# Fit a simple linear regression.model_simple = LinearRegression()model_simple.fit(X, y)# Predictions for the regression line.y_pred = model_simple.predict(X)# Metrics.mse = mean_squared_error(y, y_pred)r2 = r2_score(y, y_pred)print(f"Slope (m):     {model_simple.coef_[0]:.6f}")print(f"Intercept (b): {model_simple.intercept_:.2f}")print(f"MSE:           {mse:.4f}")print(f"R²:            {r2:.4f}")

What do these numbers mean?- The **slope** tells you how much diabetes prevalence changes for each dollar increase  in median household income. A negative slope confirms the downward trend.- The **intercept** is the model's prediction when income is zero — not meaningful in  practice, but needed to position the line.- **MSE** measures the average squared error. Lower is better.- **R²** tells you the fraction of the variation in diabetes prevalence that the model  explains. A value of 1.0 would mean a perfect fit; 0.0 means the model explains nothing.Let's plot the regression line on top of the scatter plot to see how it looks.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))# Scatter plot of actual data.ax.scatter(X, y, alpha=0.6, edgecolors="steelblue", facecolors="lightblue",           linewidths=0.8, label="Actual")# Regression line.income_range = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)ax.plot(income_range, model_simple.predict(income_range),        color="red", linewidth=2, label=f"Regression line (R²={r2:.3f})")ax.set_xlabel("Median Household Income ($)", fontsize=12)ax.set_ylabel("Diabetes Prevalence (%)", fontsize=12)ax.set_title("Linear Regression: Diabetes Prevalence vs. Income\nCalifornia Counties",             fontsize=14)ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))ax.legend(fontsize=11)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()

### Residuals: What the Model MissesThe vertical distance between each dot and the regression line is the **residual** — theerror for that county. Plotting the residuals helps us see patterns the model is missing.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))residuals = y - y_predax.scatter(y_pred, residuals, alpha=0.6, edgecolors="steelblue",           facecolors="lightblue", linewidths=0.8)ax.axhline(0, color="red", linewidth=1.5, linestyle="--")ax.set_xlabel("Predicted Diabetes Prevalence (%)", fontsize=12)ax.set_ylabel("Residual (Actual − Predicted)", fontsize=12)ax.set_title("Residual Plot — Simple Linear Regression", fontsize=14)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()

If the linear model were a perfect fit, the residuals would be randomly scattered aroundthe zero line with no visible pattern. Any pattern in the residuals — a curve, a funnelshape, clusters — tells us the model is missing something.**Key bridge to neural networks:** This idea of "measure how wrong you are, then adjust"is the fundamental principle behind how neural networks learn. A single linear regressionis, in a very real sense, a one-neuron neural network with no activation function. We'llbuild on this idea throughout the chapter.

---## 3.4 Multiple Regression: More Than One FactorDiabetes isn't driven by income alone. The correlation matrix in section 3.2 showed thatsmoking, obesity, and other factors also correlate with diabetes prevalence. Multipleregression lets us include all of these predictors at once. The model becomes:$$y = w_1 x_1 + w_2 x_2 + w_3 x_3 + \cdots + b$$Each predictor gets its own coefficient (or **weight**), and the model combines them intoa single prediction. Each weight tells us how much the outcome changes when that predictorincreases by one unit, *holding all other predictors constant*.**Key bridge to neural networks:** This is directly analogous to how a single neuroncombines multiple weighted inputs. The jump from multiple regression to a neural networkis smaller than you might think.

In [ ]:
# Predictors: income, obesity, smoking, and preventable hospital stays.PREDICTORS = {    "v063_rawvalue": "Median Household Income",    "v011_rawvalue": "Adult Obesity Rate",    "v009_rawvalue": "Adult Smoking Rate",    "v005_rawvalue": "Preventable Hospital Stays",}# Build a clean dataframe for California, dropping rows with any missing predictor.multi_data = state_data[["county"] + list(PREDICTORS.keys()) + [DIABETES_COL]].dropna().copy()X_multi = multi_data[list(PREDICTORS.keys())].valuesy_multi = multi_data[DIABETES_COL].values * 100  # percentage# Fit a multiple regression.model_multi = LinearRegression()model_multi.fit(X_multi, y_multi)y_pred_multi = model_multi.predict(X_multi)r2_multi = r2_score(y_multi, y_pred_multi)mse_multi = mean_squared_error(y_multi, y_pred_multi)print("Multiple Regression Results")print("=" * 50)for name, coef in zip(PREDICTORS.values(), model_multi.coef_):    print(f"  {name:30s}  weight = {coef:+.6f}")print(f"  {'Intercept':30s}  bias   = {model_multi.intercept_:+.4f}")print(f"\n  MSE: {mse_multi:.4f}")print(f"  R²:  {r2_multi:.4f}")print(f"\nCompare to simple regression R²: {r2:.4f}")

The R² value should be higher than our simple regression, meaning the model explainsmore of the variation in diabetes prevalence when it accounts for multiple factors. Lookat the weights — which factor has the largest effect? Which direction does each push?This lines up with what public health researchers call the **social determinants ofhealth**: the conditions in which people live, work, and age shape their health outcomesin powerful, measurable ways.

In [ ]:
# Visualize: actual vs. predicted for the multiple regression model.fig, ax = plt.subplots(figsize=(8, 6))ax.scatter(y_multi, y_pred_multi, alpha=0.6, edgecolors="steelblue",           facecolors="lightblue", linewidths=0.8)# Perfect-prediction line.lims = [min(y_multi.min(), y_pred_multi.min()) - 0.5,        max(y_multi.max(), y_pred_multi.max()) + 0.5]ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")ax.set_xlabel("Actual Diabetes Prevalence (%)", fontsize=12)ax.set_ylabel("Predicted Diabetes Prevalence (%)", fontsize=12)ax.set_title("Multiple Regression: Actual vs. Predicted\nCalifornia Counties",             fontsize=14)ax.legend(fontsize=11)ax.grid(True, alpha=0.3)ax.set_aspect("equal")ax.set_xlim(lims)ax.set_ylim(lims)plt.tight_layout()plt.show()

Points that fall close to the red dashed line are counties where the model's predictionis very accurate. Points far from the line are counties where other unmeasured factorsare influencing diabetes prevalence. No model captures everything — but multipleregression gives us a much richer picture than a single predictor alone.

---## 3.5 When Lines Aren't Enough: Polynomial RegressionSo far, all of our models have been linear — they draw straight lines (or flat planes)through the data. But many real-world relationships curve. To see this, we'll stepoutside the County Health Rankings dataset for a moment and look at how healthcarespending varies with age.The **Medical Expenditure Panel Survey (MEPS)**, conducted by the Agency for HealthcareResearch and Quality (AHRQ), collects detailed data on healthcare utilization and costsfor individuals across the United States. It's one of the most comprehensive sources ofdata on what Americans spend on health care.### Downloading the MEPS Data1. Visit the MEPS data page at [meps.ahrq.gov/mepsweb/data_stats/download_data_files.jsp](https://meps.ahrq.gov/mepsweb/data_stats/download_data_files.jsp)2. Under **Full-Year Consolidated Data Files**, find the most recent year available3. Download the **Data File (CSV)** version4. Place the downloaded CSV file in the `working/` folder next to this notebookThe key variables we need are:- `AGE##X` or `AGELAST` — the person's age- `TOTEXP##` — total healthcare expenditures for the year(The `##` is replaced by the two-digit year, e.g., `AGE22X` and `TOTEXP22` for 2022 data.)> **Note:** If the column names differ slightly in your download, the cell below will> help you identify the correct ones.

In [ ]:
# Look for the MEPS consolidated data file in the working/ folder.meps_files = sorted(glob.glob("working/*consolidated*.csv") +                    glob.glob("working/*h*.csv") +                    glob.glob("working/*MEPS*.csv"))if not meps_files:    raise FileNotFoundError(        "No MEPS consolidated data CSV found in the working/ folder.\n"        "Download it from meps.ahrq.gov and place it in 'working/'."    )meps_path = meps_files[-1]print(f"Loading: {meps_path}")meps_raw = pd.read_csv(meps_path, encoding="latin-1")print(f"Dataset shape: {meps_raw.shape[0]:,} rows × {meps_raw.shape[1]} columns")# Find the age and expenditure columns (they vary by year).age_col = Noneexp_col = Nonefor col in meps_raw.columns:    if col.startswith("AGE") and col.endswith("X"):        age_col = col    elif col == "AGELAST":        age_col = col    elif col.startswith("TOTEXP"):        exp_col = colprint(f"\nAge column:         {age_col}")print(f"Expenditure column: {exp_col}")

In [ ]:
# Clean the MEPS data: keep adults (18+), drop negative expenditures# (which MEPS uses as sentinel values for missing data).meps = meps_raw[[age_col, exp_col]].copy()meps.columns = ["age", "expenditure"]meps = meps[(meps["age"] >= 18) & (meps["expenditure"] >= 0)].dropna()print(f"Records after filtering: {len(meps):,}")print(f"Age range: {meps['age'].min()} to {meps['age'].max()}")print(f"Expenditure range: ${meps['expenditure'].min():,.0f} to ${meps['expenditure'].max():,.0f}")

### Visualizing Age vs. ExpenditureLet's start by plotting the data. With thousands of individual records, a scatter plotwould be a mess of overlapping dots. Instead, we'll compute the average expenditurefor each age and plot that.

In [ ]:
# Average expenditure by age.age_spending = meps.groupby("age")["expenditure"].mean().reset_index()fig, ax = plt.subplots(figsize=(10, 6))ax.scatter(age_spending["age"], age_spending["expenditure"],           alpha=0.7, edgecolors="steelblue", facecolors="lightblue", linewidths=0.8)ax.set_xlabel("Age", fontsize=12)ax.set_ylabel("Average Annual Healthcare Expenditure ($)", fontsize=12)ax.set_title("Healthcare Spending by Age (MEPS)", fontsize=14)ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()

The relationship is clearly not a straight line — spending is relatively flat foryounger adults, rises through middle age, and accelerates for older adults. A linearmodel would miss this curvature entirely.### Fitting Polynomial ModelsPolynomial regression extends linear regression by adding powers of the predictor.Instead of $y = mx + b$, we fit $y = ax^2 + bx + c$ (quadratic), or$y = ax^3 + bx^2 + cx + d$ (cubic), and so on.

In [ ]:
from sklearn.preprocessing import PolynomialFeaturesfrom sklearn.pipeline import make_pipeline# Use the age-averaged data for cleaner visualization.X_age = age_spending["age"].values.reshape(-1, 1)y_spend = age_spending["expenditure"].values# Fit models of increasing complexity.degrees = [1, 2, 3]models_poly = {}colors_poly = ["gray", "orange", "red"]fig, ax = plt.subplots(figsize=(10, 6))ax.scatter(X_age, y_spend, alpha=0.7, edgecolors="steelblue",           facecolors="lightblue", linewidths=0.8, label="Actual", zorder=5)x_smooth = np.linspace(X_age.min(), X_age.max(), 200).reshape(-1, 1)for deg, color in zip(degrees, colors_poly):    pipe = make_pipeline(PolynomialFeatures(deg), LinearRegression())    pipe.fit(X_age, y_spend)    y_hat = pipe.predict(x_smooth)    r2 = pipe.score(X_age, y_spend)    models_poly[deg] = pipe    label = f"Degree {deg} (R²={r2:.3f})"    ax.plot(x_smooth, y_hat, color=color, linewidth=2, label=label)ax.set_xlabel("Age", fontsize=12)ax.set_ylabel("Average Annual Expenditure ($)", fontsize=12)ax.set_title("Polynomial Regression: Healthcare Spending by Age", fontsize=14)ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))ax.legend(fontsize=10)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()

### The Danger of OverfittingA higher-degree polynomial can wiggle through the data more closely — but that isn'talways a good thing. Let's see what happens with a very high-degree polynomial.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))ax.scatter(X_age, y_spend, alpha=0.7, edgecolors="steelblue",           facecolors="lightblue", linewidths=0.8, label="Actual", zorder=5)for deg, color, ls in [(3, "red", "-"), (10, "purple", "--"), (20, "green", ":")]:    pipe = make_pipeline(PolynomialFeatures(deg), LinearRegression())    pipe.fit(X_age, y_spend)    y_hat = pipe.predict(x_smooth)    r2 = pipe.score(X_age, y_spend)    ax.plot(x_smooth, y_hat, color=color, linewidth=2, linestyle=ls,            label=f"Degree {deg} (R²={r2:.3f})")ax.set_xlabel("Age", fontsize=12)ax.set_ylabel("Average Annual Expenditure ($)", fontsize=12)ax.set_title("Overfitting: High-Degree Polynomials Chase Noise", fontsize=14)ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))ax.legend(fontsize=10)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()

Notice how the high-degree polynomials (dashed and dotted lines) swing wildly, especiallyat the edges of the age range. They achieve a higher R² on the training data by memorizingnoise rather than learning the true underlying pattern. This is **overfitting** — the modelfits the training data so closely that it would give terrible predictions on new data.**Key bridge to neural networks:** Neural networks solve the non-linearity problem using**activation functions** rather than polynomial terms. The key insight is the same: real-worldrelationships are rarely straight lines, so we need tools that can bend. But we also needto guard against bending too much. We'll return to this tension in section 3.7.

---## 3.6 How Models Learn: Gradient DescentSo far, we've been using scikit-learn to fit our models. But how does the fitting actuallywork? In this section, we'll implement the core learning algorithm — **gradient descent** —from scratch, using nothing but Python and NumPy.We'll go back to our California data: predicting diabetes prevalence from median householdincome. This is the same simple linear regression from section 3.3, but this time we'llwatch the model learn step by step.### The IdeaGradient descent works like this:1. **Start with a guess.** Pick random values for the slope and intercept.2. **Measure how wrong you are.** Compute the mean squared error.3. **Figure out which direction to adjust.** Compute the *gradient* — how the error changes   when you nudge each parameter.4. **Take a small step in the right direction.** Adjust the slope and intercept to reduce   the error.5. **Repeat** until the error stops decreasing.**Key bridge to neural networks:** This is exactly the same algorithm that trains neuralnetworks. The only difference is scale — a simple regression has 2 parameters; a largeneural network has millions. The principle is identical.

In [ ]:
# We'll work with the California data from section 3.1.# Normalize the features so gradient descent converges nicely.X_gd = state_data[INCOME_COL].values.copy()y_gd = state_data[DIABETES_COL].values * 100  # percentage# Feature scaling: subtract mean, divide by std.X_mean, X_std = X_gd.mean(), X_gd.std()y_mean, y_std = y_gd.mean(), y_gd.std()X_norm = (X_gd - X_mean) / X_stdy_norm = (y_gd - y_mean) / y_std# Initialize parameters randomly.np.random.seed(42)m = np.random.randn()  # slopeb = np.random.randn()  # intercept# Hyperparameters.learning_rate = 0.1n_iterations = 50n = len(X_norm)# Track the history for visualization.history = {"iteration": [], "m": [], "b": [], "mse": []}for i in range(n_iterations):    # Forward pass: make predictions.    y_hat = m * X_norm + b    # Compute loss (MSE).    error = y_hat - y_norm    mse = (error ** 2).mean()    # Compute gradients.    dm = (2 / n) * (error * X_norm).sum()    db = (2 / n) * error.sum()    # Update parameters.    m -= learning_rate * dm    b -= learning_rate * db    # Record history.    history["iteration"].append(i)    history["m"].append(m)    history["b"].append(b)    history["mse"].append(mse)print(f"Final MSE (normalized): {history['mse'][-1]:.6f}")print(f"Converged after {n_iterations} iterations")

### Watching the Model LearnLet's plot the regression line at several points during training to see how it evolvesfrom a random guess toward the best fit.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))snapshots = [0, 5, n_iterations - 1]for ax, idx in zip(axes, snapshots):    ax.scatter(X_norm, y_norm, alpha=0.5, edgecolors="steelblue",               facecolors="lightblue", linewidths=0.6, s=30)    m_snap = history["m"][idx]    b_snap = history["b"][idx]    x_line = np.array([X_norm.min(), X_norm.max()])    ax.plot(x_line, m_snap * x_line + b_snap, "r-", linewidth=2)    ax.set_title(f"Iteration {idx}\nMSE = {history['mse'][idx]:.4f}", fontsize=12)    ax.set_xlabel("Income (normalized)")    ax.set_ylabel("Diabetes Prevalence (normalized)")    ax.grid(True, alpha=0.3)plt.suptitle("Gradient Descent: Watching the Model Learn", fontsize=14, y=1.02)plt.tight_layout()plt.show()

### The Loss LandscapeWe can also visualize the loss as a function of the slope and intercept. Gradient descentwalks downhill on this surface toward the minimum.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))ax.plot(history["iteration"], history["mse"], "o-", color="steelblue",        markersize=4, linewidth=1.5)ax.set_xlabel("Iteration", fontsize=12)ax.set_ylabel("Mean Squared Error", fontsize=12)ax.set_title("Loss Curve: MSE Over Training Iterations", fontsize=14)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()

### Comparing to scikit-learnLet's confirm that our hand-built gradient descent arrived at the same answer asscikit-learn's closed-form solution.

In [ ]:
# Convert our normalized parameters back to the original scale.m_original = m * (y_std / X_std)b_original = y_mean + b * y_std - m_original * X_meanprint("Gradient Descent (our implementation):")print(f"  Slope:     {m_original:.6f}")print(f"  Intercept: {b_original:.4f}")print()print("scikit-learn (closed-form solution):")print(f"  Slope:     {model_simple.coef_[0]:.6f}")print(f"  Intercept: {model_simple.intercept_:.4f}")print()diff_m = abs(m_original - model_simple.coef_[0])diff_b = abs(b_original - model_simple.intercept_)print(f"Difference in slope:     {diff_m:.8f}")print(f"Difference in intercept: {diff_b:.8f}")

The values should be very close (though not perfectly identical, since gradient descentis an iterative approximation while scikit-learn uses an exact algebraic solution). Thesmall remaining difference could be reduced further by running more iterations or usinga smaller learning rate.The important takeaway: you've now seen the same algorithm that powers modern neuralnetwork training, just applied to two parameters instead of millions.

---## 3.7 Knowing What You Don't Know: Measuring Model QualityA model that fits the training data well isn't necessarily a good model. We saw this withpolynomial regression in section 3.5 — a high-degree polynomial can achieve a near-perfectR² on the training data while giving wild predictions on new data.In this section, we'll learn the standard techniques for honestly evaluating model quality:train/test splits, cross-validation, and the bias-variance tradeoff. We'll apply them toour California county data from section 3.4.

### Train/Test SplitThe simplest approach: hold out some of the data for testing. Train the model on oneportion, then evaluate it on data it has never seen.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score# Use the same multiple regression setup from section 3.4.X_eval = multi_data[list(PREDICTORS.keys())].valuesy_eval = multi_data[DIABETES_COL].values * 100# Split: 70% training, 30% testing.X_train, X_test, y_train, y_test = train_test_split(    X_eval, y_eval, test_size=0.3, random_state=42)print(f"Training set: {len(X_train)} counties")print(f"Test set:     {len(X_test)} counties")# Fit on training data only.model_eval = LinearRegression()model_eval.fit(X_train, y_train)# Evaluate on both sets.r2_train = model_eval.score(X_train, y_train)r2_test = model_eval.score(X_test, y_test)print(f"\nR² on training data: {r2_train:.4f}")print(f"R² on test data:     {r2_test:.4f}")

If the model is working well, the training and test R² values should be in the sameballpark. A big gap — high training R², low test R² — is a sign of overfitting.With only 58 California counties and a 70/30 split, the test set is small (~17 counties),so the estimate can be noisy. Cross-validation gives us a more stable picture.

### Cross-ValidationInstead of a single split, cross-validation divides the data into *k* equal parts (folds).It trains the model *k* times, each time holding out a different fold for testing andtraining on the rest. The result is *k* different R² scores, whose average gives a morereliable estimate of model quality.

In [ ]:
# 5-fold cross-validation on the full California dataset.cv_scores = cross_val_score(LinearRegression(), X_eval, y_eval, cv=5, scoring="r2")print("Cross-validation R² scores (5 folds):")for i, score in enumerate(cv_scores, 1):    print(f"  Fold {i}: {score:.4f}")print(f"\nMean R²:  {cv_scores.mean():.4f}")print(f"Std dev:  {cv_scores.std():.4f}")

The mean cross-validation R² is a more honest estimate of how well the model wouldperform on new California counties. The standard deviation tells you how much theperformance varies depending on which counties end up in the test fold.### The Bias-Variance TradeoffThis is one of the most important concepts in all of statistics and machine learning.Let's see it in action by comparing models of different complexity on the same data.

In [ ]:
from sklearn.preprocessing import PolynomialFeaturesfrom sklearn.pipeline import make_pipeline# We'll use diabetes prevalence vs. income (single predictor) to make# the comparison easy to visualize.X_bv = state_data[[INCOME_COL]].valuesy_bv = state_data[DIABETES_COL].values * 100degrees_to_test = [1, 2, 3, 5, 10]results = []for deg in degrees_to_test:    pipe = make_pipeline(PolynomialFeatures(deg), LinearRegression())    # Train on all data (for training R²).    pipe.fit(X_bv, y_bv)    r2_all = pipe.score(X_bv, y_bv)    # Cross-validated R² (honest estimate).    cv = cross_val_score(pipe, X_bv, y_bv, cv=5, scoring="r2")    results.append({        "Degree": deg,        "Training R²": round(r2_all, 4),        "CV Mean R²": round(cv.mean(), 4),        "CV Std": round(cv.std(), 4),    })results_df = pd.DataFrame(results)print(results_df.to_string(index=False))

In [ ]:
# Visualize the bias-variance tradeoff.fig, ax = plt.subplots(figsize=(9, 6))ax.plot(results_df["Degree"], results_df["Training R²"],        "o-", color="steelblue", linewidth=2, markersize=8, label="Training R²")ax.plot(results_df["Degree"], results_df["CV Mean R²"],        "s--", color="coral", linewidth=2, markersize=8, label="Cross-Validation R²")ax.set_xlabel("Polynomial Degree (Model Complexity)", fontsize=12)ax.set_ylabel("R²", fontsize=12)ax.set_title("The Bias-Variance Tradeoff\nDiabetes Prevalence vs. Income — California Counties",             fontsize=14)ax.legend(fontsize=11)ax.grid(True, alpha=0.3)ax.set_xticks(degrees_to_test)plt.tight_layout()plt.show()

As model complexity increases (higher polynomial degree):- **Training R² goes up** — the model memorizes the training data more closely.- **Cross-validation R² may go up at first, then drops** — beyond a point, the model  starts fitting noise rather than real patterns.The sweet spot is where cross-validation R² is highest. A model that is too simple(**high bias**) underfits — it can't capture the real pattern. A model that is toocomplex (**high variance**) overfits — it captures noise that won't generalize.**Key bridge to neural networks:** These same evaluation techniques — train/test splits,cross-validation, watching for overfitting — are used identically in deep learning. Thereader is building habits they'll use throughout the book.

---## 3.8 Putting It Together: Not Everything Needs AILet's return to where we started. Imagine you work at the California Department ofHealth and you've been asked: *"Which counties should we prioritize for diabetesprevention programs?"*With the tools from this chapter, you can build a practical, interpretable answerwithout any neural networks or AI. Let's use our multiple regression model to identifythe counties where diabetes prevalence is higher than the model predicts — these arethe counties where something beyond the measured factors may be driving up diabetes, andwhere targeted intervention might have the most impact.

In [ ]:
# Refit the multiple regression on all California counties.model_final = LinearRegression()model_final.fit(X_multi, y_multi)predicted = model_final.predict(X_multi)residuals_final = y_multi - predicted# Build a summary table.summary = multi_data[["county"]].copy()summary.columns = ["County"]summary["Actual Diabetes (%)"] = y_multi.round(2)summary["Predicted Diabetes (%)"] = predicted.round(2)summary["Residual (%)"] = residuals_final.round(2)summary = summary.sort_values("Residual (%)", ascending=False).reset_index(drop=True)print("Counties where diabetes is HIGHER than the model predicts")print("(positive residual = potential priority for intervention)")print("=" * 65)print(summary.head(10).to_string(index=False))print()print("\nCounties where diabetes is LOWER than the model predicts")print("(negative residual = potential bright spots to learn from)")print("=" * 65)print(summary.tail(10).to_string(index=False))

In [ ]:
# Visualize the residuals on a bar chart.fig, ax = plt.subplots(figsize=(14, 6))colors = ["coral" if r > 0 else "steelblue" for r in summary["Residual (%)"]]ax.barh(range(len(summary)), summary["Residual (%)"], color=colors, edgecolor="white")ax.set_yticks(range(len(summary)))ax.set_yticklabels(summary["County"], fontsize=8)ax.set_xlabel("Residual: Actual − Predicted Diabetes Prevalence (%)", fontsize=12)ax.set_title(    "California Counties: Diabetes Prevalence vs. Model Prediction\n"    "Coral = higher than expected | Blue = lower than expected",    fontsize=13,)ax.axvline(0, color="black", linewidth=0.8)ax.grid(True, alpha=0.3, axis="x")ax.invert_yaxis()plt.tight_layout()plt.show()

### When Do You Need AI?The regression model we built is simple, fast, interpretable, and useful. A public healthanalyst can explain exactly why the model flagged a particular county — it's not a blackbox. For many population health questions, this is exactly the right tool.You might need something more powerful — a neural network, a random forest, or anothermachine learning model — when:- The relationships in your data are highly non-linear and interactions between variables  are complex (we saw a taste of this with polynomial regression).- You have very large datasets with hundreds or thousands of features.- You're working with unstructured data like medical images, clinical notes, or genomic  sequences.- The patterns in the data are too subtle for a human to specify in a formula.In the chapters ahead, we'll encounter exactly these situations and build the tools tohandle them. But the foundation you've built here — loss functions, gradient descent,model evaluation, and the bias-variance tradeoff — will carry through everything thatfollows.

---### Chapter 3 SummaryIn this chapter, you learned:1. **Descriptive statistics** — how to summarize and visualize population health data   using means, medians, histograms, and correlation matrices (Section 3.2).2. **Linear regression** — how to model the relationship between a predictor and an   outcome, and what slope, intercept, MSE, and R² mean (Section 3.3).3. **Multiple regression** — how to include several predictors at once, with each   getting its own weight — directly analogous to how a neuron combines inputs   (Section 3.4).4. **Polynomial regression** — how to model non-linear relationships, and the danger   of overfitting when the model is too complex (Section 3.5).5. **Gradient descent** — the algorithm that powers both classical regression and modern   neural networks: measure the error, compute the gradient, take a step (Section 3.6).6. **Model evaluation** — train/test splits, cross-validation, and the bias-variance   tradeoff (Section 3.7).These concepts are the building blocks of machine learning. In the next chapter, we'llsee how they extend into the world of neural networks.